# EDA - press-cross features (`features.cross.build_press_cross_features`)

Validates the 8 pressure-cross feature recommendations from the press-EDA.
The features themselves are implemented in `features/cross.py`; this
notebook re-loads them, profiles their distributions, evaluates their
bivariate signal vs `scored_after`, tests redundancy and significance,
and finalises a keep / drop manifest.

| # | Feature | Source mechanism |
|---|---------|------------------|
| 1 | `top_third_presence_joint`                        | `top_third_run_share x top_third_press_share` (multiplicative gate) |
| 2 | `top_third_press_share_x_set_piece_share`         | extends shots-EDA's strongest interaction |
| 3 | `pressing_minus_pressed_x_cumul_shots`            | tactical-role disambiguator |
| 4 | `press_turnover_rate_{A,M,D,G}`                   | role-conditional composure failure |
| 5 | `forward_pass_share_{A,M,D,G}`                    | role-conditional progressive intent |
| 6 | `top_third_press_share_above_team_avg`            | team-relative spatial press exposure |
| 7 | `last15_press_events_x_last15_intensity_shots`    | strongest cross-domain recency surge |
| 8 | `top_third_presence_avg`                          | additive complement to the joint gate |

Section structure:

| ID | Section |
|----|---------|
| A | Build & profile (univariate description) |
| B | Bivariate signal vs `scored_after` (pooled + by position) |
| C | Redundancy check (correlation among the 14 columns) |
| D | Cluster-robust GLM + BH-FDR significance |
| E | Final manifest |


## Setup

In [1]:
"""Imports."""
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT: Path = Path.cwd().parent if Path.cwd().name == "eda" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from features.cross import build_press_cross_features, PRESS_CROSS_FEATURE_COLUMNS


In [2]:
"""Display options."""
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 80)
pd.set_option("display.precision", 4)


In [3]:
"""Load source CSVs."""
DATA_DIR = PROJECT_ROOT / "data"
main  = pd.read_csv(DATA_DIR / "players_quarters_final.csv")
runs  = pd.read_csv(DATA_DIR / "player_appearance_run.csv")
shots = pd.read_csv(DATA_DIR / "player_appearance_shot_limited.csv")
press = pd.read_csv(DATA_DIR / "player_appearance_behaviour_under_pressure.csv")
main.shape, runs.shape, shots.shape, press.shape


((3486, 33), (35133, 10), (780, 12), (12185, 10))

In [4]:
"""Helpers (Wilson CI + binned-rate table)."""
from scipy import stats as _stats


def wilson_ci(successes: int, trials: int, alpha: float = 0.05) -> tuple[float, float]:
    if trials == 0:
        return float("nan"), float("nan")
    p = successes / trials
    z = _stats.norm.ppf(1.0 - alpha / 2.0)
    denom = 1.0 + (z ** 2) / trials
    centre = (p + (z ** 2) / (2.0 * trials)) / denom
    half = z * np.sqrt(p * (1.0 - p) / trials + (z ** 2) / (4.0 * trials ** 2)) / denom
    return float(centre - half), float(centre + half)


def binned_rate(df: pd.DataFrame, feature: str, target: str = "scored_after",
                bins=None, by: str | None = None) -> pd.DataFrame:
    work = df[[feature, target] + ([by] if by else [])].copy()
    if bins is not None:
        work["__bin"] = pd.cut(work[feature], bins=bins, include_lowest=True)
    else:
        work["__bin"] = work[feature]
    grouping = ["__bin"] if by is None else [by, "__bin"]
    g = work.groupby(grouping, observed=False, dropna=False)[target].agg(["size", "sum"])
    g = g.rename(columns={"size": "n", "sum": "n_pos"}).reset_index()
    g["rate"] = g["n_pos"] / g["n"].where(g["n"] > 0)
    ci = g.apply(lambda r: wilson_ci(int(r["n_pos"]), int(r["n"])), axis=1, result_type="expand")
    g["ci_lo"], g["ci_hi"] = ci[0], ci[1]
    return g.rename(columns={"__bin": "bin"})


## Section A - Build & profile

In [5]:
"""Build the press-cross feature frame."""
cross = build_press_cross_features(main, runs, shots, press)
print(f"shape: {cross.shape}")
assert cross.shape[0] == len(main), "row count must match the main panel"
assert list(cross.columns) == list(PRESS_CROSS_FEATURE_COLUMNS)
cross.head(3)


shape: (3486, 16)


,player_appearance_id,checkpoint,top_third_presence_joint,top_third_press_share_x_set_piece_share,pressing_minus_pressed_x_cumul_shots,press_turnover_rate_A,press_turnover_rate_M,press_turnover_rate_D,press_turnover_rate_G,forward_pass_share_A,forward_pass_share_M,forward_pass_share_D,forward_pass_share_G,top_third_press_share_above_team_avg,last15_press_events_x_last15_intensity_shots,top_third_presence_avg
0,39421,H1_15,NaN,NaN,0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,NaN,NaN,0.0,NaN
1,39422,H1_15,0.0,NaN,0,0.0,0.0,0.5,0.0,0.0,0.0,1.0,0.0,-0.0312,0.0,0.0
2,39423,H1_15,NaN,NaN,0,0.0,0.0,NaN,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0


In [6]:
"""Univariate description."""
NUMERIC = [c for c in cross.columns if c not in ("player_appearance_id", "checkpoint")]
desc = cross[NUMERIC].describe().T[["count", "mean", "std", "min", "50%", "max"]]
desc["n_nan"] = len(cross) - desc["count"].astype(int)
desc.round(4)


,count,mean,std,min,50%,max,n_nan
top_third_presence_joint,3179.0,0.1004,0.1345,0.0000,0.0476,1.0000,307
top_third_press_share_x_set_piece_share,1008.0,0.0614,0.1473,0.0000,0.0000,1.0000,2478
pressing_minus_pressed_x_cumul_shots,3486.0,-0.2298,8.8409,-120.0000,0.0000,72.0000,0
press_turnover_rate_A,3463.0,0.0877,0.2181,0.0000,0.0000,1.0000,23
press_turnover_rate_M,3445.0,0.1369,0.2219,0.0000,0.0000,1.0000,41
press_turnover_rate_D,3440.0,0.1212,0.2163,0.0000,0.0000,1.0000,46
press_turnover_rate_G,3348.0,0.0229,0.1335,0.0000,0.0000,1.0000,138
forward_pass_share_A,3399.0,0.0393,0.1463,0.0000,0.0000,1.0000,87
forward_pass_share_M,3385.0,0.1117,0.2152,0.0000,0.0000,1.0000,101
forward_pass_share_D,3385.0,0.1466,0.2668,0.0000,0.0000,1.0000,101


### A - findings (interpretation)

Sanity-check expectations:

* `top_third_presence_joint` and `top_third_presence_avg` are bounded in
  `[0, 1]`. The joint version is mostly zero (one of two factors is zero),
  the average version is broader.
* `pressing_minus_pressed_x_cumul_shots` has a wide signed range -
  positive = high-press attacker (pressing more than pressed AND
  shooting), negative = pressed playmaker.
* The 4 position-dummy families (`press_turnover_rate_{A,M,D,G}`,
  `forward_pass_share_{A,M,D,G}`) are mostly zero by construction
  (one row activates one of the four positions).
* `top_third_press_share_above_team_avg` should average to ~0 by
  construction (deviation from team mean per fixture x side x cp).
* NaN counts are non-zero on every ratio whose denominator can be 0.
  The downstream model's NaN policy decides imputation.


## Section B - Bivariate signal vs `scored_after`

Pearson r ranking and binned target rates with Wilson CI.


In [7]:
"""Merge for analysis."""
m = main[["player_appearance_id", "checkpoint", "scored_after",
          "fixture_id", "position"]].merge(
    cross, on=["player_appearance_id", "checkpoint"], how="left"
)
BASELINE = m["scored_after"].mean()
print(f"baseline scored_after rate: {BASELINE:.4f}")


baseline scored_after rate: 0.0582


In [8]:
"""Pearson r vs target."""
rows = []
for c in NUMERIC:
    s = m[c].dropna()
    if s.std() == 0:
        rows.append({"feature": c, "n": len(s), "pearson_r": float("nan")})
        continue
    t = m.loc[s.index, "scored_after"]
    rows.append({"feature": c, "n": len(s), "pearson_r": float(np.corrcoef(s, t)[0, 1])})
pearson_df = pd.DataFrame(rows).sort_values(
    "pearson_r", key=lambda x: x.abs(), ascending=False
).reset_index(drop=True)
pearson_df


,feature,n,pearson_r
0,press_turnover_rate_A,3463,0.1346
1,top_third_presence_joint,3179,0.1258
2,top_third_press_share_x_set_piece_share,1008,0.1237
3,top_third_presence_avg,3422,0.1186
4,top_third_press_share_above_team_avg,3238,0.1015
5,forward_pass_share_D,3385,-0.0828
6,last15_press_events_x_last15_intensity_shots,3486,0.0707
7,press_turnover_rate_D,3440,-0.0697
8,forward_pass_share_A,3399,0.0657
9,press_turnover_rate_G,3348,-0.0437


In [9]:
"""Binned target rates with Wilson CIs."""
BINS = {
    "top_third_presence_joint":                     [-0.01, 0, 0.05, 0.2, 1.01],
    "top_third_press_share_x_set_piece_share":      [-0.01, 0, 0.1, 0.3, 1.01],
    "pressing_minus_pressed_x_cumul_shots":         [-200, -5, 0, 5, 200],
    "press_turnover_rate_A":                        [-0.01, 0, 0.2, 0.5, 1.01],
    "press_turnover_rate_M":                        [-0.01, 0, 0.2, 0.5, 1.01],
    "press_turnover_rate_D":                        [-0.01, 0, 0.2, 0.5, 1.01],
    "press_turnover_rate_G":                        [-0.01, 0, 1.01],
    "forward_pass_share_A":                         [-0.01, 0, 0.2, 0.5, 1.01],
    "forward_pass_share_M":                         [-0.01, 0, 0.2, 0.5, 1.01],
    "forward_pass_share_D":                         [-0.01, 0, 0.2, 0.5, 1.01],
    "forward_pass_share_G":                         [-0.01, 0, 1.01],
    "top_third_press_share_above_team_avg":         [-1, -0.1, 0.1, 1],
    "last15_press_events_x_last15_intensity_shots": [-0.01, 0, 0.2, 0.5, 5],
    "top_third_presence_avg":                       [-0.01, 0.1, 0.25, 0.5, 1.01],
}

frames = []
for feat, edges in BINS.items():
    t = binned_rate(m, feat, "scored_after", bins=edges)
    t["feature"] = feat
    frames.append(t)
binned_df = pd.concat(frames, ignore_index=True)[
    ["feature", "bin", "n", "n_pos", "rate", "ci_lo", "ci_hi"]
]
binned_df


,feature,bin,n,n_pos,rate,ci_lo,ci_hi
0,top_third_presence_joint,"(-0.011, 0.0]",1085,49,0.0452,3.4328e-02,0.0592
1,top_third_presence_joint,"(0.0, 0.05]",534,22,0.0412,2.7362e-02,0.0616
2,top_third_presence_joint,"(0.05, 0.2]",992,49,0.0494,3.7563e-02,0.0647
3,top_third_presence_joint,"(0.2, 1.01]",568,71,0.1250,1.0030e-01,0.1547
4,top_third_presence_joint,NaN,307,12,0.0391,2.2499e-02,0.0671
5,top_third_press_share_x_set_piece_share,"(-0.011, 0.0]",783,48,0.0613,4.6547e-02,0.0803
6,top_third_press_share_x_set_piece_share,"(0.0, 0.1]",30,1,0.0333,5.9086e-03,0.1667
7,top_third_press_share_x_set_piece_share,"(0.1, 0.3]",125,16,0.1280,8.0347e-02,0.1978
8,top_third_press_share_x_set_piece_share,"(0.3, 1.01]",70,11,0.1571,9.0076e-02,0.2599
9,top_third_press_share_x_set_piece_share,NaN,2478,127,0.0513,4.3242e-02,0.0606


### B - findings (interpretation)

Patterns we expect from the prior reasoning:

* `top_third_presence_joint` and `top_third_presence_avg` should both
  show steep monotone climbs to >10% at the top bin (vs 5.8% baseline).
* `top_third_press_share_above_team_avg` should be monotone: pressed
  more than teammates in attacking third = elevated rate.
* `pressing_minus_pressed_x_cumul_shots` likely **non-monotone**: both
  positive and negative extremes elevate (high-press attacker vs
  pressed playmaker). Tree-friendly only.
* Position-dummy families: only the position-active row carries info;
  signs may differ across positions (M and D usually positive on
  turnover_rate via the attacker-confound; A and G near-zero).


## Section C - Redundancy among the 14 columns

By construction:

* `top_third_presence_joint` and `top_third_presence_avg` share the
  same two parents - high collinearity expected.
* `top_third_press_share_x_set_piece_share` shares `top_third_press_share`
  with the team-relative version.
* The position-dummy families internally are orthogonal across positions
  but multicollinear with the position one-hot encoding.


In [10]:
"""Pearson correlation matrix on the 14 numeric columns."""
corr = cross[NUMERIC].corr(method="pearson")
hi = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .rename("pearson")
        .reset_index()
        .rename(columns={"level_0": "a", "level_1": "b"})
)
hi["abs"] = hi["pearson"].abs()
hi.sort_values("abs", ascending=False).head(15).reset_index(drop=True)


,a,b,pearson,abs
0,top_third_presence_joint,top_third_presence_avg,0.9108,0.9108
1,top_third_press_share_above_team_avg,top_third_presence_avg,0.8385,0.8385
2,top_third_presence_joint,top_third_press_share_above_team_avg,0.7743,0.7743
3,press_turnover_rate_G,forward_pass_share_G,0.6060,0.6060
4,press_turnover_rate_M,forward_pass_share_M,0.5814,0.5814
5,press_turnover_rate_D,forward_pass_share_D,0.5789,0.5789
6,press_turnover_rate_A,forward_pass_share_A,0.5575,0.5575
7,press_turnover_rate_A,top_third_presence_avg,0.3593,0.3593
8,press_turnover_rate_M,press_turnover_rate_D,-0.3515,0.3515
9,press_turnover_rate_M,forward_pass_share_D,-0.3489,0.3489


### C - findings (interpretation)

Decision logic:

* Pairs with `|pearson| >= 0.95` -> drop the dependent partner.
* For pairs in the 0.7-0.95 range, keep both but note that linear models
  will distribute the coefficient mass across them; tree models will
  pick whichever splits cleaner.
* The two `top_third_presence_*` features are expected to correlate
  strongly - decide which form to keep based on D-section significance.


## Section D - Cluster-robust significance + BH-FDR

In [11]:
"""Cluster-robust GLM + BH-FDR."""
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests


def cluster_robust_p(df: pd.DataFrame, feature: str) -> tuple[float, float, int]:
    sub = df[[feature, "scored_after", "fixture_id"]].dropna()
    if sub[feature].std() == 0 or len(sub) < 30:
        return float("nan"), float("nan"), len(sub)
    try:
        fitted = smf.logit(f"scored_after ~ {feature}", data=sub).fit(
            disp=False, maxiter=200,
            cov_type="cluster", cov_kwds={"groups": sub["fixture_id"]},
        )
        return float(fitted.params[feature]), float(fitted.pvalues[feature]), len(sub)
    except Exception:
        return float("nan"), float("nan"), len(sub)


rows = []
for f in NUMERIC:
    coef, p, n = cluster_robust_p(m, f)
    r = m[[f, "scored_after"]].corr().iloc[0, 1]
    rows.append({"feature": f, "n": n, "pearson_r": r, "coef": coef, "p_raw": p})
sig = pd.DataFrame(rows)

mask = sig["p_raw"].notna()
adj = np.full(len(sig), np.nan)
if mask.sum():
    _, adj_p, _, _ = multipletests(sig.loc[mask, "p_raw"].values, method="fdr_bh")
    adj[mask.values] = adj_p
sig["p_bh"] = adj
sig["bh_q05"] = sig["p_bh"] < 0.05
sig.sort_values("p_bh").reset_index(drop=True)


C:\Users\tymot\projects\wec\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\tymot\projects\wec\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
C:\Users\tymot\projects\wec\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,feature,n,pearson_r,coef,p_raw,p_bh,bh_q05
0,forward_pass_share_G,3281,-0.0436,-37.8356,2.3983e-149,3.1178e-148,True
1,press_turnover_rate_A,3463,0.1346,1.7930,7.6584e-07,4.9780e-06,True
2,top_third_presence_joint,3179,0.1258,2.7689,1.5073e-06,6.5315e-06,True
3,top_third_presence_avg,3422,0.1186,2.2684,1.0053e-05,3.2671e-05,True
4,last15_press_events_x_last15_intensity_shots,3486,0.0707,1.3263,4.7599e-04,1.2376e-03,True
5,top_third_press_share_above_team_avg,3238,0.1015,1.6072,2.9163e-03,5.4748e-03,True
6,top_third_press_share_x_set_piece_share,1008,0.1237,2.1357,2.9480e-03,5.4748e-03,True
7,forward_pass_share_A,3399,0.0657,1.3331,3.6276e-03,5.8949e-03,True
8,forward_pass_share_D,3385,-0.0828,-2.0243,2.6245e-02,3.7909e-02,True
9,press_turnover_rate_D,3440,-0.0697,-1.9449,5.3546e-02,6.9609e-02,False


### D - findings (interpretation)

Three outcome buckets:

1. **Survives BH q=0.05 with rate above baseline -> KEEP unconditionally.**
2. **BH-significant but rate ~ baseline -> drop (perfect-separation
   artefact, likely a goalkeeper-dummy column).**
3. **Bivariate-significant Pearson but BH non-significant -> demote
   unless tree feature-importance later rescues it.**

Key features expected to survive: `top_third_presence_avg`,
`top_third_presence_joint`, `top_third_press_share_above_team_avg`,
`last15_press_events_x_last15_intensity_shots`.


## Section E - Final manifest

In [12]:
"""Programmatic manifest with robust verdict."""
def best_non_baseline_bin(df: pd.DataFrame, feature: str) -> tuple[float, float, float, int]:
    sub = df[df["feature"] == feature]
    if sub.empty:
        return (float("nan"),) * 3 + (0,)
    sub = sub.sort_values("rate", ascending=False).iloc[0]
    return float(sub["rate"]), float(sub["ci_lo"]), float(sub["ci_hi"]), int(sub["n"])


rows = []
for feat_name in NUMERIC:
    rate, lo, hi, n = best_non_baseline_bin(binned_df, feat_name)
    rows.append({"feature": feat_name, "best_rate": rate,
                 "ci_lo": lo, "ci_hi": hi, "n": n})

manifest = pd.DataFrame(rows).merge(
    sig[["feature", "pearson_r", "p_bh", "bh_q05"]], on="feature", how="left"
)


def verdict(row) -> str:
    """Robust verdict: ignore spurious BH p_bh = 0 from perfect separation."""
    if pd.isna(row["best_rate"]):
        return "drop (no data)"
    rate_above = row["best_rate"] > BASELINE * 1.2
    ci_above = (not pd.isna(row["ci_lo"]) and row["ci_lo"] > BASELINE)
    if row["bh_q05"] is True and (rate_above or ci_above):
        return "KEEP"
    if ci_above:
        return "keep (cautious)"
    if row["best_rate"] > BASELINE * 1.4 and row["n"] >= 30:
        return "keep (cautious)"
    return "drop (flat)"


manifest["verdict"] = manifest.apply(verdict, axis=1)
manifest.sort_values(["verdict", "p_bh"]).reset_index(drop=True)


,feature,best_rate,ci_lo,ci_hi,n,pearson_r,p_bh,bh_q05,verdict
0,press_turnover_rate_A,0.2609,0.1255,0.4647,23,0.1346,4.9780e-06,True,KEEP
1,top_third_presence_joint,0.1250,0.1003,0.1547,568,0.1258,6.5315e-06,True,KEEP
2,top_third_presence_avg,0.1160,0.0891,0.1497,431,0.1186,3.2671e-05,True,KEEP
3,last15_press_events_x_last15_intensity_shots,0.1273,0.0630,0.2402,55,0.0707,1.2376e-03,True,KEEP
4,top_third_press_share_x_set_piece_share,0.1571,0.0901,0.2599,70,0.1237,5.4748e-03,True,KEEP
5,top_third_press_share_above_team_avg,0.0955,0.0790,0.1149,1037,0.1015,5.4748e-03,True,KEEP
6,forward_pass_share_A,0.2529,0.1733,0.3533,87,0.0657,5.8949e-03,True,KEEP
7,forward_pass_share_D,0.0730,0.0633,0.0840,2426,-0.0828,3.7909e-02,True,KEEP
8,forward_pass_share_G,0.0637,0.0557,0.0727,3187,-0.0436,3.1178e-148,True,drop (flat)
9,forward_pass_share_M,0.0792,0.0407,0.1486,101,0.0341,2.8147e-01,False,drop (flat)


### Closing notes

* This manifest is the deliverable. Compare against `eda_cross_features.ipynb`
  to see how runs-cross and pressure-cross features stack against each
  other before merging into the final modelling table.
* For RQ4 ("does pressure data help?"), the cluster-robust significance
  table in Section D directly answers it: every feature that survives
  with rate-above-baseline is incremental signal that the no-pressure
  baseline does not have.
* Cross features that fail BH still have a chance to rescue themselves
  via tree-model feature importance during the modelling step
  (especially the non-monotone `pressing_minus_pressed_x_cumul_shots`).
